In [0]:
# Importação de funções nativas do PySpark para manipulação de DataFrames
# current_timestamp: gera a data e hora atuais do momento da execução
# col: permite referenciar colunas específicas do DataFrame (incluindo metadados)
from pyspark.sql.functions import current_timestamp, col

# 1. DEFINIÇÃO DO CAMINHO DE ORIGEM (UNITY CATALOG VOLUMES)
# Define o diretório onde os arquivos CSV brutos estão armazenados
VOLUME_PATH = "/Volumes/workspace/mvp_puc_dados/mvp_olist_raw/"

# Executa um comando SQL diretamente na sessão Spark para garantir que o banco de dados/schema existe
# 'IF NOT EXISTS' previne que o código falhe caso a base 'bronze_olist' já tenha sido criada no Unity Catalog
spark.sql("CREATE DATABASE IF NOT EXISTS bronze_olist")

# 2. MAPEAMENTO DE DADOS (DICIONÁRIO DE/PARA)
# Estrutura de dados chave-valor associando o nome do arquivo fonte (CSV) ao nome de destino da tabela Bronze
datasets = {
    "olist_customers_dataset.csv": "bronze_customers",
    "olist_orders_dataset.csv": "bronze_orders",
    "olist_order_items_dataset.csv": "bronze_order_items",
    "olist_order_payments_dataset.csv": "bronze_order_payments",
    "olist_products_dataset.csv": "bronze_products",
    "olist_sellers_dataset.csv": "bronze_sellers"
}

# 3. LOOP DE INGESTÃO E TRANSFORMAÇÃO (CSV -> DELTA LAKE)
# Itera sobre cada par (chave, valor) do dicionário
for csv_file, table_name in datasets.items():
    # Interpola o caminho base com o arquivo específico para compor o caminho completo de leitura
    file_full_path = f"{VOLUME_PATH}/{csv_file}"
    
    print(f"Lendo e processando: {csv_file} -> Tabela: bronze_olist.{table_name}")
    
    # --- ETAPA 1: LEITURA DO ARQUIVO CSV ---
    df = (spark.read
          .option("header", "true")
          .option("inferSchema", "true")
          .csv(file_full_path))
    
    # --- ETAPA 2: ADIÇÃO DE METADADOS DE AUDITORIA ---
    # Cria a coluna '_ingestion_time' com o timestamp exato do processamento (rastreabilidade temporal)
    # Cria a coluna '_source_file' usando o campo oculto '_metadata.file_path' do Databricks para registrar a origem exata do dado
    df_bronze = df.withColumn("_ingestion_time", current_timestamp()) \
                  .withColumn("_source_file", col("_metadata.file_path"))
    
    # --- ETAPA 3: ESCRITA NA CAMADA BRONZE (FORMATO DELTA LAKE) ---
    (df_bronze.write
     .format("delta")
     .mode("overwrite")
     .option("overwriteSchema", "true")
     .saveAsTable(f"bronze_olist.{table_name}"))

print("\nIngestão da Camada Bronze concluída com sucesso!")

In [0]:
display(dbutils.fs.ls("/Volumes/workspace/mvp_puc_dados/mvp_olist_raw/"))